In [1]:
from datasets import load_dataset

og_dataset = load_dataset("sentence-transformers/natural-questions", cache_dir="datasets", split="train")
dataset = og_dataset.shuffle(seed=22).select(range(1000))

In [7]:
from query_processing.missing import missing
from query_processing.missclicks import missclicks
from query_processing.moved import moved
from query_processing.process_query import process_query

configs = [
    {"mapping": missclicks, "ratios": [0.01, 0.05, 0.1, 0.15, 0.2]},
    {"mapping": missing, "ratios": [0.01, 0.03, 0.05, 0.07, 0.1]},
    {"mapping": moved, "ratios": [0.01, 0.03, 0.05, 0.07, 0.1]}
]

process_query(dataset, configs)

In [ ]:
from completions.complete import complete
from completions.evaluate import evaluate
from completions.get_predictions import get_predictions

base_dir= "datasets/raw_datasets"
save_dir = "datasets/predictions"                 

get_predictions(base_dir, save_dir, complete)

base_dir= "datasets/predictions"
save_dir = "datasets/evaluations"     

get_predictions(base_dir, save_dir, evaluate)

In [9]:
import os
from datasets import load_from_disk
from datasets import Dataset

base_dir= "datasets/evaluations"

for base_datasets in os.listdir(base_dir):
    base_datasets_path = f"{base_dir}/{base_datasets}"
    for dataset in os.listdir(base_datasets_path):
        base_dataset_path = f"{base_datasets_path}/{dataset}"
        
        loaded_dataset = load_from_disk(base_dataset_path)

        empty = 0
        failed = 0
        success = 0
        confident = 0
        for example in loaded_dataset:
            if example["raw_judge"]=="":
                empty+=1
            if example["json_failed"]==True:
                failed+=1
            if example["judge_label"]=="correct":
                success+=1
            if example["confidence"]:
                if example["confidence"]>=80:
                    confident+=1
        print(base_dataset_path)
        #print(f"Empty: {empty/len(loaded_dataset)}")
        #print(f"Failed json: {failed/len(loaded_dataset)}")
        print(f"Successes: {success/len(loaded_dataset)}")
        print(f"Confident: {confident/len(loaded_dataset)}")

datasets/evaluations/baseline/baseline_hf
Successes: 0.47539267015706804
Confident: 0.9214659685863874
datasets/evaluations/missclicks/ratio_0.01
Successes: 0.453091684434968
Confident: 0.9125799573560768
datasets/evaluations/missclicks/ratio_0.05
Successes: 0.41430073606729756
Confident: 0.917981072555205
datasets/evaluations/missclicks/ratio_0.1
Successes: 0.3798941798941799
Confident: 0.9005291005291005
datasets/evaluations/missclicks/ratio_0.15
Successes: 0.296137339055794
Confident: 0.8572961373390557
datasets/evaluations/missclicks/ratio_0.2
Successes: 0.25496688741721857
Confident: 0.8554083885209713
datasets/evaluations/missing/ratio_0.01
Successes: 0.4707724425887265
Confident: 0.9363256784968684
datasets/evaluations/missing/ratio_0.03
Successes: 0.44656084656084655
Confident: 0.9238095238095239
datasets/evaluations/missing/ratio_0.05
Successes: 0.42344244984160506
Confident: 0.9218585005279831
datasets/evaluations/missing/ratio_0.07
Successes: 0.3871657754010695
Confident: 0.